# Fase 1 — Meta-analisi Omici IR-AKI

## Obiettivo
Identificare geni e pathway consistentemente alterati nel danno renale
da ischemia-riperfusione (IR-AKI), attraverso una meta-analisi multi-dataset.

---

## Stato del progetto

| Fase | Stato | Note |
|------|-------|------|
| **GSE43974** (questo notebook) | ✅ Completato | 1269 seed genes, filtro bilaterali applicato |
| GSE142077 → GSE53769 → GSE90861 → GSE126805 | ⬜ Da fare | Categoria A, meta-analisi Fisher in R |
| Meta-analisi cross-dataset | ⬜ Da fare | Dopo ≥3 dataset Cat. A |
| GSE54888 → GSE37838 → GSE293480 | ⬜ Da fare | Categoria B, validazione severità |
| Fase 2 — Network Medicine | ⬜ Da fare | PPI, hub genes, network propagation |

---

## Cosa fa questo notebook

**Dataset:** GSE43974 — Damman et al., *Transplantation* 2015
Illumina HumanHT-12 V4, 554 biopsie renali (523 dopo filtro bilaterali)

**Preprocessing:**
1. Parsing metadata dal series matrix + filtro 31 biopsie rene destro (pseudo-replicazione)
2. Caricamento non-normalized GenomeStudio (AVG_Signal + Detection Pval)
3. Filtro detection → log2 → quantile normalization

**Analisi DE (Welch t-test, FDR BH < 0.05, |log2FC| > 0.379):**
- **Principale:** Deceased_T3 (n=169) vs Living_T1 (n=37) → seed genes per meta-analisi
- **Supporto:** Deceased_T1 (n=120) vs Living_T1 (n=37) → flag `ischemia_also`

**Output:** `de_GSE43974_IRI_vs_Control.csv`, `seed_genes_GSE43974.csv`, `ranking_GSE43974.csv`

---

## Correzioni applicate rispetto alla versione iniziale

- **Filtro biopsie bilaterali:** rimossi 31 campioni rene destro (19 BD + 12 DCD al T1)
  usando la colonna GEO `"deceased left and right biopsies same donor (0=left, 1=right)"`.
  I gruppi ora corrispondono esattamente all'articolo (BD T1=82, DCD T1=38).
- **Confronto supporto:** corretto da 151 vs 37 a 120 vs 37 (niente pseudo-replicazione).
- **Confronto principale:** non affetto (i T3 e Living T1 non avevano bilaterali).

---

## Prossimo passo

**GSE142077** — Park et al., BMC Nephrol 2020
- RNA-seq, 5 pazienti × 3 condizioni (pre-ischemia, ischemia, reperfusione 10min), design paired
- Confronto: R10 vs P (reperfusione vs pre-ischemia)
- Pipeline: assemblare counts da Salmon quant.sf → pyDESeq2 → `de_GSE142077_IRI_vs_Control.csv`

# Fase 1 — Meta-analisi Omici IR-AKI

## Obiettivo
Identificare geni e pathway consistentemente alterati nel danno renale
da ischemia-riperfusione (IR-AKI), attraverso una meta-analisi multi-dataset.

## Strategia generale
Per ogni dataset si esegue:
1. **Preprocessing** dei dati raw (log2 + quantile normalization per microarray, DESeq2 per RNA-seq)
2. **Espressione differenziale**: un confronto principale **IRI vs Control**
3. **Probe → Gene mapping** (per microarray)
4. **Salvataggio** del risultato gene-level (log2FC, pvalue, padj)

I risultati gene-level di tutti i dataset vengono poi combinati in una
**meta-analisi cross-dataset** con il metodo di Fisher in R.

---

## Pipeline per GSE43974 (questo notebook)

### Dati
- **Piattaforma:** Illumina HumanHT-12 V4 (GPL10558), 47323 sonde
- **File usati:** `GSE43974_non-normalized_data.txt.gz` + `GSE43974_series_matrix.txt.gz`
- **Articolo:** Damman et al., *Transplantation* 2015

### Design dello studio
554 biopsie renali a 3 timepoint (523 dopo rimozione 31 biopsie bilaterali rene destro):

| Timepoint | Living | DBD | DCD | Significato |
|-----------|--------|-----|-----|-------------|
| T1 (pre-retrieval) | 37 (controllo sano) | 82 (post-brain death) | 38 (post-cardiac arrest) | Baseline / ischemia |
| T2 (fine cold ischemia) | — | 110 | 53 | Ischemia fredda |
| T3 (post-riperfusione) | 34 | 105 | 64 | IRI completa |

> **Nota:** Il GEO (GSE43974) contiene 554 campioni, di cui 31 sono biopsie
> del rene destro da donatori con biopsia bilaterale. L'articolo usa solo il
> rene sinistro ("Only the left biopsy was included in the aforementioned analyses").
> Questi 31 campioni vengono filtrati nel preprocessing del metadata.

### Selezione campioni per la meta-analisi

| Confronto | Case | Control | Razionale |
|-----------|------|---------|-----------|
| **Principale** | Deceased_T3 (DBD+DCD, n=169) | Living_T1 (n=37) | IRI completa vs rene sano |
| **Supporto** | Deceased_T1 (DBD+DCD, n=120) | Living_T1 (n=37) | Componente ischemica sola |

Il confronto principale è quello usato per la meta-analisi cross-dataset.
Il confronto supporto serve per annotare quali geni sono alterati già nella fase
ischemica (prima della riperfusione).

### Preprocessing (replica dell'articolo)
1. **Parsing** del file non-normalized GenomeStudio (AVG_Signal + Detection Pval)
2. **Filtro bilaterali:** rimozione 31 biopsie rene destro (colonna GEO `deceased left and right biopsies same donor`)
3. **Filtro detection:** rimosse sonde mai detected (p<0.05) in nessun campione
4. **Log2 transform:** `log2(intensity)` con floor a 1
5. **Quantile normalization:** distribuzioni rese identiche tra campioni

### Soglie DE
- |log2FC| > 0.379 (equivale a FC ≥ 1.3, come nell'articolo)
- FDR (Benjamini-Hochberg) < 0.05

### Output prodotti
| File | Contenuto |
|------|-----------|
| `de_GSE43974_IRI_vs_Control.csv` | DE gene-level, confronto principale |
| `de_GSE43974_Ischemia_vs_Control.csv` | DE gene-level, confronto supporto |
| `seed_genes_GSE43974.csv` | Geni significativi (con flag `ischemia_also`) |
| `ranking_GSE43974.csv` | log2FC |

In [1]:
# Setup
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import rankdata
from IPython.display import display

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "geo"
OUTPUT_DIR   = PROJECT_ROOT / "output" / "fase1"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NON_NORM_FILE   = DATA_DIR / "GSE43974" / "GSE43974_non-normalized_data.txt.gz"
SERIES_MAT_FILE = DATA_DIR / "GSE43974" / "GSE43974_series_matrix.txt.gz"

FC_THRESHOLD  = 0.379   # |log2FC| ≥ log2(1.3)
FDR_THRESHOLD = 0.05

print(f"Non-norm file: {NON_NORM_FILE}  (exists: {NON_NORM_FILE.exists()})")
print(f"Series matrix: {SERIES_MAT_FILE}  (exists: {SERIES_MAT_FILE.exists()})")

Non-norm file: /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/data/geo/GSE43974/GSE43974_non-normalized_data.txt.gz  (exists: True)
Series matrix: /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/data/geo/GSE43974/GSE43974_series_matrix.txt.gz  (exists: True)


 ## 1. Metadata dal Series Matrix

 Estraiamo GSM IDs, titoli e caratteristiche dall'header del file
 series_matrix.txt.gz. Questo ci dà il mapping posizionale
 chip_col[i] → GSM_ID[i] per rinominare le colonne del non-normalized.

In [2]:
sample_ids, sample_titles = [], []


# Gestisce il caso in cui diverse subset di campioni hanno
# chiavi diverse nella stessa riga !Sample_characteristics_ch1

sample_chars_raw = []  # lista di liste, una per riga

with pd.io.common.get_handle(SERIES_MAT_FILE, mode="r", compression="gzip") as f:
    for line in f.handle:
        if line.startswith("!Sample_geo_accession"):
            sample_ids = [s.strip('"') for s in line.strip().split("\t")[1:]]
        elif line.startswith("!Sample_title"):
            sample_titles = [s.strip('"') for s in line.strip().split("\t")[1:]]
        elif line.startswith("!Sample_characteristics_ch1"):
            parts = [s.strip('"') for s in line.strip().split("\t")[1:]]
            sample_chars_raw.append(parts)
        elif line.startswith("!series_matrix_table_begin"):
            break

print(f"Campioni: {len(sample_ids)}")
print(f"Righe characteristics: {len(sample_chars_raw)}")

Campioni: 554
Righe characteristics: 6


In [3]:
# Costruisci metadata campione per campione
meta = pd.DataFrame({"title": sample_titles}, index=sample_ids)
meta.index.name = "gsm_id"

# Per ogni riga di characteristics, estrai key:value per ogni campione
for row_idx, row in enumerate(sample_chars_raw):
    for sample_idx, val in enumerate(row):
        if ":" in val:
            key = val.split(":")[0].strip()
            value = val.split(":", 1)[1].strip()
            col_name = key
            if col_name not in meta.columns:
                meta[col_name] = np.nan
            meta.iloc[sample_idx, meta.columns.get_loc(col_name)] = value

# Parse donor_type e timepoint
meta["donor_type"] = meta["title"].str.extract(r"Kidney_donor_(\w+)_T")
meta["timepoint"]  = meta["title"].str.extract(r"_T(\d)_").astype(float)

# ──────────────────────────────────────────────────────────────────────────
# FILTRO BIOPSIE BILATERALI (rene destro)
# Damman et al.: "From 31 of the 120 deceased donors [...] we also collected
# biopsies of the left and right kidney from the same donor. Only the left
# biopsy was included in the aforementioned analyses."
#
# La colonna GEO:
#   "deceased left and right biopsies same donor (0=left, 1=right)"
#   Valore 1 = rene destro → va escluso per evitare pseudo-replicazione.
# ──────────────────────────────────────────────────────────────────────────
bilateral_col = "deceased left and right biopsies same donor (0=left, 1=right)"

if bilateral_col in meta.columns:
    meta[bilateral_col] = pd.to_numeric(meta[bilateral_col], errors="coerce")
    right_kidney_mask = meta[bilateral_col] == 1.0
    n_right = right_kidney_mask.sum()
    print(f"Biopsie rene destro identificate: {n_right}")
    print(f"  BD T1:  {((meta['donor_type']=='BD')  & (meta['timepoint']==1) & right_kidney_mask).sum()}")
    print(f"  DCD T1: {((meta['donor_type']=='DCD') & (meta['timepoint']==1) & right_kidney_mask).sum()}")

    meta = meta[~right_kidney_mask].copy()
    print(f"Campioni dopo rimozione reni destri: {len(meta)}  (rimossi {n_right})")
else:
    print(f"ATTENZIONE: colonna '{bilateral_col}' non trovata — nessun filtro applicato")

# DGF
dgf_cols = [c for c in meta.columns if "delayed graft function" in c.lower()]
if dgf_cols:
    meta["dgf"] = meta[dgf_cols].bfill(axis=1).iloc[:, 0]
    meta["dgf"] = pd.to_numeric(meta["dgf"], errors="coerce")
    meta.loc[~meta["dgf"].isin([0.0, 1.0]), "dgf"] = np.nan

print(f"\nDGF per gruppo:")
display(meta.groupby(["donor_type", "timepoint"])["dgf"].value_counts(dropna=False).unstack(fill_value=0))

Biopsie rene destro identificate: 31
  BD T1:  19
  DCD T1: 12
Campioni dopo rimozione reni destri: 523  (rimossi 31)

DGF per gruppo:


dgf                   0.0  1.0  NaN
donor_type timepoint               
BD         1.0         51   22    9
           2.0         81   29    0
           3.0         69   36    0
DCD        1.0         18   20    0
           2.0         13   40    0
           3.0         11   53    0
Living     1.0         34    2    1
           3.0         31    2    1

In [4]:
# Verifica gruppi (devono corrispondere all'articolo Damman et al. 2015)
print("\nGruppi donor_type × timepoint (dopo filtro bilaterali):")
ct = meta.groupby(["donor_type", "timepoint"]).size().unstack(fill_value=0)
display(ct)

# Check automatico contro i numeri dell'articolo
expected = {"BD": {1: 82, 2: 110, 3: 105},
            "DCD": {1: 38, 2: 53, 3: 64},
            "Living": {1: 37, 3: 34}}
all_ok = True
for dt, tp_dict in expected.items():
    for tp, n_exp in tp_dict.items():
        n_obs = ct.loc[dt, float(tp)] if dt in ct.index else 0
        status = "✓" if n_obs == n_exp else f"✗ (atteso {n_exp})"
        if n_obs != n_exp:
            all_ok = False
        print(f"  {dt} T{tp}: {n_obs} {status}")

if all_ok:
    print("\n✓ Tutti i gruppi corrispondono all'articolo Damman et al. 2015")
else:
    print("\n✗ ATTENZIONE: alcuni gruppi non corrispondono!")


Gruppi donor_type × timepoint (dopo filtro bilaterali):


timepoint,1.0,2.0,3.0
donor_type,,,
BD,82,110,105
DCD,38,53,64
Living,37,0,34


  BD T1: 82 ✓
  BD T2: 110 ✓
  BD T3: 105 ✓
  DCD T1: 38 ✓
  DCD T2: 53 ✓
  DCD T3: 64 ✓
  Living T1: 37 ✓
  Living T3: 34 ✓

✓ Tutti i gruppi corrispondono all'articolo Damman et al. 2015


### DGF (Delayed Graft Function)
La colonna DGF è presente per la maggior parte dei campioni.
Dopo la rimozione delle 31 biopsie bilaterali (rene destro),
i gruppi T1 corrispondono esattamente all'articolo (BD=82, DCD=38, Living=37).

In [5]:
dgf_cols = [c for c in meta.columns if "delayed graft function" in c.lower()]
print(f"Colonne DGF trovate: {dgf_cols}")

if dgf_cols:
    print(f"\nDGF value counts (NaN = non disponibile):")
    display(meta["dgf"].value_counts(dropna=False))

    print(f"\nDGF per gruppo (dove disponibile):")
    dgf_avail = meta[meta["dgf"].notna()]
    display(dgf_avail.groupby(["donor_type", "timepoint", "dgf"]).size().unstack(fill_value=0))
else:
    meta["dgf"] = np.nan
    print("NESSUNA colonna DGF trovata!")

Colonne DGF trovate: ['delayed graft function (0=no, 1=yes)']

DGF value counts (NaN = non disponibile):


dgf
0.0    308
1.0    204
NaN     11
Name: count, dtype: int64


DGF per gruppo (dove disponibile):


dgf                   0.0  1.0
donor_type timepoint          
BD         1.0         51   22
           2.0         81   29
           3.0         69   36
DCD        1.0         18   20
           2.0         13   40
           3.0         11   53
Living     1.0         34    2
           3.0         31    2

## 2. Caricamento Non-Normalized Data
Il file `GSE43974_non-normalized_data.txt.gz` ha formato GenomeStudio:
- Header Illumina (righe `[Header]`, `[Sample Probe Profile]`, ecc.)
- Poi la tabella dati con colonne: TargetID, PROBE_ID, SYMBOL, ...
  e per ogni campione: chipID.AVG_Signal, chipID.Detection Pval, ecc.
Dobbiamo trovare dove inizia la tabella dati.

In [6]:
skip_rows = 0
with pd.io.common.get_handle(NON_NORM_FILE, mode="r", compression="gzip") as f:
    for i, line in enumerate(f.handle):
        print(f"Riga {i}: {line.strip()}")
        if line.startswith("[Sample Probe Profile]"):
            skip_rows = i + 1
            break
        if i > 50:
            break

print(f"Righe header da saltare: {skip_rows}")

Riga 0: [Header]
Riga 1: GSGX Version	1.9.0
Riga 2: Report Date	3/8/2012 11:57
Riga 3: Project	JD_All-Raw
Riga 4: Group Set	JD_All-Raw
Riga 5: Analysis	JD_All-Raw_nonorm_nobkgd
Riga 6: Normalization	none
Riga 7: [Sample Probe Profile]
Righe header da saltare: 8


In [7]:
print("Caricamento (può richiedere ~1 min)...")
df_raw = pd.read_csv(
    NON_NORM_FILE, sep="\t", compression="gzip",
    index_col=0, skiprows=skip_rows, low_memory=False
)

# ====================================================================
# PATCH: Excel date-gene corruption fix
#
# Ref 1: Ziemann et al., Genome Biol 2016;17:177
#        doi:10.1186/s13059-016-1044-7
# Ref 2: Bruford et al., Nat Genet 2020;52:754-758
#        doi:10.1038/s41588-020-0669-3 (Box 3)
# Ref 3: HGNC symbol reports consultati 2026-04-08:
#        BHLHE40 (HGNC:1046), BHLHE41 (HGNC:16617)
#        https://www.genenames.org/
#
# Nomenclatura HGNC 2020+ per coerenza cross-dataset.
# ====================================================================
excel_fix = {
    # MARCH → MARCHF (Bruford 2020, Box 3)
    "1-Mar": "MARCHF1", "2-Mar": "MARCHF2", "3-Mar": "MARCHF3",
    "4-Mar": "MARCHF4", "5-Mar": "MARCHF5", "6-Mar": "MARCHF6",
    "7-Mar": "MARCHF7", "8-Mar": "MARCHF8", "9-Mar": "MARCHF9",
    "10-Mar": "MARCHF10", "11-Mar": "MARCHF11",

    # SEPT → SEPTIN (Bruford 2020, Box 3)
    "1-Sep": "SEPTIN1", "2-Sep": "SEPTIN2", "3-Sep": "SEPTIN3",
    "4-Sep": "SEPTIN4", "5-Sep": "SEPTIN5", "6-Sep": "SEPTIN6",
    "7-Sep": "SEPTIN7", "8-Sep": "SEPTIN8", "9-Sep": "SEPTIN9",
    "10-Sep": "SEPTIN10", "11-Sep": "SEPTIN11", "12-Sep": "SEPTIN12",
    "13-Sep": "SEPTIN13", "14-Sep": "SEPTIN14", "15-Sep": "SEPTIN15",

    # DEC1 → BHLHE40 (HGNC:1046), DEC2 → BHLHE41 (HGNC:16617)
    "1-Dec": "BHLHE40",
    "2-Dec": "BHLHE41",
}

# 1. Correggi l'indice (TargetID)
df_raw.rename(index=excel_fix, inplace=True)

# 2. CRITICO: Correggi anche la colonna SYMBOL
if "SYMBOL" in df_raw.columns:
    df_raw["SYMBOL"] = df_raw["SYMBOL"].replace(excel_fix)
    
print("✅ Bug delle date di Excel corretto nell'indice e nella colonna SYMBOL.")
# ====================================================================
# FINE PATCH EXCEL
# ====================================================================

print(f"Shape: {df_raw.shape}")
print(f"\nPrime 5 colonne: {df_raw.columns[:5].tolist()}")
print(f"Ultime 5 colonne: {df_raw.columns[-5:].tolist()}")
print(f"\nIndice (TargetID) — primi 5: {df_raw.index[:5].tolist()}")

# %%
# Colonne di annotazione presenti
ann_cols = [c for c in ["PROBE_ID", "SYMBOL", "ILMN_GENE", "ENTREZ_GENE_ID",
                         "DEFINITION", "CHROMOSOME", "ACCESSION"] if c in df_raw.columns]
print(f"Colonne annotazione: {ann_cols}")

# Colonne dati
intensity_cols = [c for c in df_raw.columns if c.endswith(".AVG_Signal")]
pval_cols      = [c for c in df_raw.columns
                  if "Detection Pval" in c or "detection pval" in c.lower()]

print(f"Colonne AVG_Signal:     {len(intensity_cols)}")
print(f"Colonne Detection Pval: {len(pval_cols)}")
print(f"Campioni attesi:        {len(sample_ids)}")

# Verifica allineamento
assert len(intensity_cols) == len(sample_ids), \
    f"Mismatch: {len(intensity_cols)} intensity cols vs {len(sample_ids)} campioni"

Caricamento (può richiedere ~1 min)...
✅ Bug delle date di Excel corretto nell'indice e nella colonna SYMBOL.
Shape: (47323, 2225)

Prime 5 colonne: ['ProbeID', '5937311049_A.AVG_Signal', '5937311049_A.BEAD_STDERR', '5937311049_A.Avg_NBEADS', '5937311049_A.Detection Pval']
Ultime 5 colonne: ['ENTREZ_GENE_ID', 'SYMBOL', 'PROBE_ID', 'DEFINITION', 'SYNONYMS']

Indice (TargetID) — primi 5: ['7A5', 'A1BG', 'A1BG', 'A1CF', 'A1CF']
Colonne annotazione: ['PROBE_ID', 'SYMBOL', 'ILMN_GENE', 'ENTREZ_GENE_ID', 'DEFINITION']
Colonne AVG_Signal:     554
Colonne Detection Pval: 554
Campioni attesi:        554


In [8]:
# ====================================================================
# TEST DI VERIFICA (Sanity Check) SULLA PATCH EXCEL
# ====================================================================
import re  # Importa la libreria nativa di Python per le Espressioni Regolari (Regex)

print("\n🧪 TEST DI VERIFICA SULLA PATCH EXCEL...")

# 1. DEFINIZIONE DELLA REGEX
# Questa stringa funge da "identikit" matematico per scovare le false date.
# Spiegazione della formula r"^\d{1,2}-(Jan|Feb|...|Dec)$":
#   ^          : Indica l'inizio esatto della stringa. Non cerca in mezzo ad altre parole.
#   \d{1,2}    : Cerca 1 o 2 cifre numeriche (es. "1", "01", "31").
#   -          : Cerca il carattere letterale del trattino centrale.
#   (Jan|...)  : L'operatore | significa "OPPURE". Cerca uno qualsiasi di questi mesi.
#   $          : Indica la fine esatta della stringa.
#   IGNORECASE : Rende la ricerca insensibile a maiuscole/minuscole (es. trova sia "1-Dec" che "1-dec").
date_pattern = re.compile(r"^\d{1,2}-(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$", re.IGNORECASE)

# 2. ISPEZIONE DELL'INDICE (TargetID)
# Scorre ogni elemento dell'indice, lo forza a diventare testo (.astype(str)) 
# e verifica se combacia perfettamente (.match()) con il nostro identikit.
index_corrupted = [val for val in df_raw.index.dropna().astype(str) if date_pattern.match(val)]

# 3. ISPEZIONE DELLA COLONNA SYMBOL
# Applica la stessa logica di ispezione, ma controlla la colonna dei nomi 
# dei geni (assicurandosi prima che tale colonna esista nel file).
symbol_corrupted = []
if "SYMBOL" in df_raw.columns:
    symbol_corrupted = [val for val in df_raw["SYMBOL"].dropna().astype(str) if date_pattern.match(val)]

# 4. VERDETTO FINALE
# Se entrambe le liste di "sospetti" sono vuote (not), il test è superato.
if not index_corrupted and not symbol_corrupted:
    print("✅ TEST SUPERATO: Nessuna traccia di date di Excel in df_raw!")
    print("I nomi dei geni (es. DEC1, MARCH1) sono pronti e sicuri per l'analisi.")
else:
    # Se anche solo un elemento è rimasto incastrato, fa scattare l'allarme e lo stampa.
    print("❌ ATTENZIONE: Il test ha fallito. Trovate date residue:")
    if index_corrupted: 
        print(f"  -> Trovate nell'indice: {set(index_corrupted)}")
    if symbol_corrupted: 
        print(f"  -> Trovate in SYMBOL: {set(symbol_corrupted)}")
# ====================================================================


🧪 TEST DI VERIFICA SULLA PATCH EXCEL...
✅ TEST SUPERATO: Nessuna traccia di date di Excel in df_raw!
I nomi dei geni (es. DEC1, MARCH1) sono pronti e sicuri per l'analisi.


### 2.1 Probe-to-Gene Mapping (dall'annotazione embedded)
Il file contiene le colonne PROBE_ID (ILMN_*) e SYMBOL direttamente.
Le estraiamo prima di isolare la matrice numerica.

In [9]:
probe_to_gene = {}
if "PROBE_ID" in df_raw.columns and "SYMBOL" in df_raw.columns:
    for ilmn, sym in zip(df_raw["PROBE_ID"].values, df_raw["SYMBOL"].values):
        ilmn = str(ilmn).strip()
        sym  = str(sym).strip()
        if ilmn not in ("nan", "") and sym not in ("nan", "", "---", "NA"):
            probe_to_gene[ilmn] = sym

print(f"Mapping probe → gene: {len(probe_to_gene)} entries")
# Esempio
for k in list(probe_to_gene.keys())[:5]:
    print(f"  {k} → {probe_to_gene[k]}")

Mapping probe → gene: 44053 entries
  ILMN_1762337 → 7A5
  ILMN_2055271 → A1BG
  ILMN_1736007 → A1BG
  ILMN_2383229 → A1CF
  ILMN_1806310 → A1CF


### 2.2 Estrai matrice intensità e Detection P-value
Re-indicizziamo con PROBE_ID (ILMN_*) e rinominiamo le colonne
con i GSM IDs usando il mapping posizionale.

In [10]:
if "PROBE_ID" not in df_raw.columns:
    raise ValueError("Colonna PROBE_ID non trovata!")

ilmn_index = df_raw["PROBE_ID"].astype(str).str.strip().values

# Matrice intensità
df_intensity = df_raw[intensity_cols].copy()
df_intensity.index      = ilmn_index
df_intensity.index.name = "probe_id"
df_intensity.columns    = sample_ids
df_intensity = df_intensity[~df_intensity.index.isin(["nan", "", "NaN"])]
df_intensity = df_intensity.apply(pd.to_numeric, errors="coerce")

# Matrice detection p-values
df_pval = df_raw[pval_cols].copy()
df_pval.index      = ilmn_index
df_pval.index.name = "probe_id"
df_pval.columns    = sample_ids
df_pval = df_pval[~df_pval.index.isin(["nan", "", "NaN"])]
df_pval = df_pval.apply(pd.to_numeric, errors="coerce")

print(f"Intensità: {df_intensity.shape}")
print(f"Detection: {df_pval.shape}")

# Verifica scala
print(f"\nIntensità — statistiche:")
print(f"  min:    {df_intensity.values.min():.1f}")
print(f"  median: {np.nanmedian(df_intensity.values):.1f}")
print(f"  max:    {df_intensity.values.max():.1f}")

display(df_intensity.iloc[:5, :5])

Intensità: (47323, 554)
Detection: (47323, 554)

Intensità — statistiche:
  min:    67.2
  median: 118.8
  max:    31151.8


,GSM1075188,GSM1075189,GSM1075190,GSM1075191,GSM1075192
probe_id,,,,,
ILMN_1762337,115.8,135.1,120.7,121.6,125.6
ILMN_2055271,127.4,133.0,113.9,125.5,127.0
ILMN_1736007,116.5,108.1,112.1,106.8,114.5
ILMN_2383229,582.5,454.1,260.0,283.5,627.9
ILMN_1806310,824.3,676.6,335.5,346.7,919.2


In [11]:
# Anteprima detection p-values
print("Detection p-values — anteprima:")
display(df_pval.iloc[:5, :5])
print(f"\nFrazione sonde 'detected' (p<0.05) per campione (primi 5):")
for col in df_pval.columns[:5]:
    frac = (df_pval[col] < 0.05).mean()
    print(f"  {col}: {frac:.1%}")


Detection p-values — anteprima:


,GSM1075188,GSM1075189,GSM1075190,GSM1075191,GSM1075192
probe_id,,,,,
ILMN_1762337,0.25584,0.00519,0.07662,0.13636,0.13117
ILMN_2055271,0.03636,0.00649,0.20649,0.08182,0.11039
ILMN_1736007,0.24156,0.37532,0.25974,0.52727,0.41688
ILMN_2383229,0.00000,0.00000,0.00130,0.00000,0.00000
ILMN_1806310,0.00000,0.00000,0.00000,0.00000,0.00000



Frazione sonde 'detected' (p<0.05) per campione (primi 5):
  GSM1075188: 37.1%
  GSM1075189: 37.7%
  GSM1075190: 34.7%
  GSM1075191: 35.2%
  GSM1075192: 41.1%


## 3. Preprocessing
Pipeline:
1. **Filtro detection** (opzionale — l'articolo non lo applica esplicitamente,
   ma noi lo mostriamo per trasparenza. Il filtro è leggero: togliamo
   solo le sonde mai rilevate in nessun campione)
2. **Log2 transform** (con floor a 1 per evitare log di valori ≤ 0)
3. **Quantile normalization** (standard Illumina, replica GeneSpring)

In [12]:
# STEP 1: filtro detection leggero
# Rimuoviamo solo sonde mai detected in nessun campione
detected_in_any = (df_pval < 0.05).any(axis=1)
n_before = len(df_intensity)
n_never  = (~detected_in_any).sum()
print(f"Sonde totali:              {n_before}")
print(f"Mai detected (p<0.05):     {n_never}")
print(f"Mantenute:                 {n_before - n_never}")

# Applichiamo il filtro
df_intensity_filt = df_intensity[detected_in_any].copy()
print(f"\nShape dopo filtro: {df_intensity_filt.shape}")

Sonde totali:              47323
Mai detected (p<0.05):     5256
Mantenute:                 42067

Shape dopo filtro: (42067, 554)


In [13]:
# Dopo il filtro detection, rimuovi sonde non mappabili
has_gene = df_intensity_filt.index.isin(probe_to_gene.keys())
print(f"Sonde con gene symbol: {has_gene.sum()}/{len(df_intensity_filt)}")
# NON filtrare qui — lo facciamo dopo la DE nel probe-to-gene step

Sonde con gene symbol: 38811/42067


In [14]:
# STEP 2: Log2 transform
df_log2 = np.log2(df_intensity_filt.clip(lower=1))

print(f"Post log2:")
print(f"  min:    {df_log2.values.min():.2f}")
print(f"  median: {np.nanmedian(df_log2.values):.2f}")
print(f"  max:    {df_log2.values.max():.2f}")

display(df_log2.iloc[:5, :5])

Post log2:
  min:    6.07
  median: 6.94
  max:    14.93


,GSM1075188,GSM1075189,GSM1075190,GSM1075191,GSM1075192
probe_id,,,,,
ILMN_1762337,6.855491,7.077884,6.915282,6.925999,6.972693
ILMN_2055271,6.993221,7.055282,6.831624,6.971544,6.988685
ILMN_1736007,6.864186,6.756223,6.808642,6.738768,6.839204
ILMN_2383229,9.186114,8.826866,8.022368,8.147205,9.294391
ILMN_1806310,9.687026,9.402159,8.390169,8.437544,9.844235


In [15]:
# STEP 3: Quantile Normalization
def quantile_normalize(df):
    """
    Quantile normalization sulle colonne (campioni).
    Rende identica la distribuzione in tutti i campioni.
    Standard per microarray Illumina.
    """
    arr = df.values.astype(float)
    sorted_arr = np.sort(arr, axis=0)
    row_means  = sorted_arr.mean(axis=1)
    result = np.zeros_like(arr)
    for j in range(arr.shape[1]):
        col = arr[:, j].copy()
        nan_mask = np.isnan(col)
        if nan_mask.any():
            col[nan_mask] = np.nanmedian(col)
        ranks = rankdata(col, method="min").astype(int) - 1
        result[:, j] = row_means[ranks]
    return pd.DataFrame(result, index=df.index, columns=df.columns)


print(f"Quantile normalization ({df_log2.shape[0]} probes × {df_log2.shape[1]} campioni)...")
expr = quantile_normalize(df_log2)

print(f"\nPost quantile norm:")
print(f"  min:    {expr.values.min():.2f}")
print(f"  median: {np.nanmedian(expr.values):.2f}")
print(f"  max:    {expr.values.max():.2f}")

display(expr.iloc[:5, :5])

Quantile normalization (42067 probes × 554 campioni)...

Post quantile norm:
  min:    6.32
  median: 6.93
  max:    14.52


,GSM1075188,GSM1075189,GSM1075190,GSM1075191,GSM1075192
probe_id,,,,,
ILMN_1762337,6.868917,7.122094,6.991496,6.932362,6.902982
ILMN_2055271,7.021149,7.102347,6.893415,6.979315,6.917292
ILMN_1736007,6.878011,6.808962,6.868818,6.743820,6.780763
ILMN_2383229,9.211768,8.524009,8.389076,8.019295,8.715523
ILMN_1806310,9.727106,9.040121,8.813364,8.280550,9.229780


In [16]:
# Verifica: distribuzioni per campione (devono essere identiche post-QN)
print("Mediane per campione (primi 10) — devono essere identiche:")
for col in expr.columns[:10]:
    print(f"  {col}: {expr[col].median():.4f}")

Mediane per campione (primi 10) — devono essere identiche:
  GSM1075188: 6.9300
  GSM1075189: 6.9302
  GSM1075190: 6.9296
  GSM1075191: 6.9298
  GSM1075192: 6.9303
  GSM1075193: 6.9303
  GSM1075194: 6.9300
  GSM1075195: 6.9308
  GSM1075196: 6.9305
  GSM1075197: 6.9300


## 4. Definizione gruppi e verifica

In [17]:
m = meta
living_t1   = m[(m["donor_type"] == "Living") & (m["timepoint"] == 1)].index.tolist()
dbd_t1      = m[(m["donor_type"] == "BD")     & (m["timepoint"] == 1)].index.tolist()
dcd_t1      = m[(m["donor_type"] == "DCD")    & (m["timepoint"] == 1)].index.tolist()
dbd_t3      = m[(m["donor_type"] == "BD")     & (m["timepoint"] == 3)].index.tolist()
dcd_t3      = m[(m["donor_type"] == "DCD")    & (m["timepoint"] == 3)].index.tolist()

deceased_t1 = dbd_t1 + dcd_t1
deceased_t3 = dbd_t3 + dcd_t3

print(f"Living_T1  (control):    {len(living_t1)}")
print(f"DBD_T1     (ischemia):   {len(dbd_t1)}")
print(f"DCD_T1     (ischemia):   {len(dcd_t1)}")
print(f"Deceased_T1 totale:      {len(deceased_t1)}")
print(f"DBD_T3     (IRI):        {len(dbd_t3)}")
print(f"DCD_T3     (IRI):        {len(dcd_t3)}")
print(f"Deceased_T3 totale:      {len(deceased_t3)}")

# Verifica che tutti i GSM siano nella matrice di espressione
for name, ids in [("Living_T1", living_t1), ("Deceased_T3", deceased_t3)]:
    in_expr = [s for s in ids if s in expr.columns]
    print(f"\n{name}: {len(in_expr)}/{len(ids)} presenti in expr")

Living_T1  (control):    37
DBD_T1     (ischemia):   82
DCD_T1     (ischemia):   38
Deceased_T1 totale:      120
DBD_T3     (IRI):        105
DCD_T3     (IRI):        64
Deceased_T3 totale:      169

Living_T1: 37/37 presenti in expr

Deceased_T3: 169/169 presenti in expr


In [18]:
# DGF nei gruppi T3 (dove ha senso clinico)
print("\nDGF nei gruppi T3 (riperfusione):")
for dtype, name in [("BD", "DBD"), ("DCD", "DCD")]:
    t3 = m[(m["donor_type"] == dtype) & (m["timepoint"] == 3)]
    dgf_yes = (t3["dgf"] == 1).sum()
    dgf_no  = (t3["dgf"] == 0).sum()
    dgf_na  = t3["dgf"].isna().sum()
    print(f"  {name}_T3: DGF=1: {dgf_yes}, DGF=0: {dgf_no}, NA: {dgf_na}")


DGF nei gruppi T3 (riperfusione):
  DBD_T3: DGF=1: 36, DGF=0: 69, NA: 0
  DCD_T3: DGF=1: 53, DGF=0: 11, NA: 0


## 5. Espressione Differenziale
Welch t-test probe per probe, FDR con Benjamini-Hochberg.
log2FC = mean(case) − mean(ctrl) sui valori log2 quantile-normalized.

In [19]:
from statsmodels.stats.multitest import multipletests

def de_microarray(expr_df, samples_case, samples_ctrl,
                  comparison_name="", dataset="GSE43974"):
    """DE con Welch t-test su valori log2."""
    case = [s for s in samples_case if s in expr_df.columns]
    ctrl = [s for s in samples_ctrl if s in expr_df.columns]

    print(f"\n{'='*60}")
    print(f"DE: {comparison_name}")
    print(f"  Case: {len(case)}, Control: {len(ctrl)}")

    if len(case) < 2 or len(ctrl) < 2:
        print("  [SKIP]")
        return pd.DataFrame()

    results = []
    for probe in expr_df.index:
        cv = expr_df.loc[probe, case].dropna().values.astype(float)
        tv = expr_df.loc[probe, ctrl].dropna().values.astype(float)
        if len(cv) < 2 or len(tv) < 2:
            continue
        tstat, pval = stats.ttest_ind(cv, tv, equal_var=False)
        results.append({
            "probe":          probe,
            "log2FoldChange": float(cv.mean() - tv.mean()),
            "pvalue":         float(pval),
            "t_statistic":    float(tstat),
            "mean_case":      float(cv.mean()),
            "mean_ctrl":      float(tv.mean()),
        })

    df = pd.DataFrame(results).set_index("probe")
    _, df["padj"], _, _ = multipletests(df["pvalue"].fillna(1), method="fdr_bh")

    sig = df[(df["padj"] < FDR_THRESHOLD) & (df["log2FoldChange"].abs() > FC_THRESHOLD)]
    print(f"  Probe testate: {len(df)}")
    print(f"  Significative (|log2FC|>{FC_THRESHOLD}, FDR<{FDR_THRESHOLD}): "
          f"{len(sig)}  ({(sig['log2FoldChange']>0).sum()}↑ "
          f"{(sig['log2FoldChange']<0).sum()}↓)")

    df["comparison"] = comparison_name
    df["dataset"]    = dataset
    return df

In [20]:
# Confronto principale: Deceased_T3 vs Living_T1
de_main = de_microarray(
    expr, deceased_t3, living_t1,
    comparison_name="IRI_vs_Control (T3_deceased vs T1_living)"
)


DE: IRI_vs_Control (T3_deceased vs T1_living)
  Case: 169, Control: 37
  Probe testate: 42067
  Significative (|log2FC|>0.379, FDR<0.05): 1541  (901↑ 640↓)


In [21]:
# Ispezione risultati del confronto principale.
# Ispezione risultati
print("Distribuzione p-value (deve avere picco vicino a 0 se c'è segnale):")
print(de_main["pvalue"].describe())
print(f"\nDistribuzione log2FC:")
print(de_main["log2FoldChange"].describe())

# Top 20 per p-value
print(f"\nTop 20 probe per significatività:")
display(de_main.sort_values("pvalue").head(20)[
    ["log2FoldChange", "pvalue", "padj", "mean_case", "mean_ctrl"]])

Distribuzione p-value (deve avere picco vicino a 0 se c'è segnale):
count    4.206700e+04
mean     3.510311e-01
std      3.195781e-01
min      1.568710e-75
25%      3.476217e-02
50%      2.777457e-01
75%      6.223786e-01
max      9.999957e-01
Name: pvalue, dtype: float64

Distribuzione log2FC:
count    42067.000000
mean         0.000023
std          0.175648
min         -1.984850
25%         -0.030272
50%         -0.002985
75%          0.022875
max          3.543909
Name: log2FoldChange, dtype: float64

Top 20 probe per significatività:


,log2FoldChange,pvalue,padj,mean_case,mean_ctrl
probe,,,,,
ILMN_1754332,1.638703,1.568710e-75,6.599092e-71,8.666714,7.028011
ILMN_1689294,1.839263,3.637196e-69,7.650296e-65,8.819493,6.980230
ILMN_1682636,1.917954,6.433462e-65,9.021215e-61,8.888767,6.970814
ILMN_1658383,1.905736,1.148331e-58,1.207671e-54,8.886299,6.980563
ILMN_1776998,2.249539,2.219984e-58,1.867761e-54,10.106705,7.857167
ILMN_1779857,2.180290,2.238246e-55,1.569272e-51,9.526787,7.346497
ILMN_1656011,1.419350,8.651127e-55,5.198957e-51,8.452227,7.032877
ILMN_2222317,1.407566,1.238823e-50,6.514197e-47,9.049608,7.642042
ILMN_1806165,2.703289,5.564322e-50,2.600826e-46,9.839544,7.136255


In [22]:
de_support = de_microarray(
    expr, deceased_t1, living_t1,
    comparison_name="Ischemia_vs_Control (T1_deceased vs T1_living)"
)


DE: Ischemia_vs_Control (T1_deceased vs T1_living)
  Case: 120, Control: 37
  Probe testate: 42067
  Significative (|log2FC|>0.379, FDR<0.05): 1013  (420↑ 593↓)


In [23]:
print(f"\nTop 20 probe ischemia:")
display(de_support.sort_values("pvalue").head(20)[
    ["log2FoldChange", "pvalue", "padj", "mean_case", "mean_ctrl"]])


Top 20 probe ischemia:


,log2FoldChange,pvalue,padj,mean_case,mean_ctrl
probe,,,,,
ILMN_1788874,3.445321,3.921636e-31,1.649715e-26,11.170814,7.725493
ILMN_2379599,1.129332,3.522239e-29,7.408500e-25,8.070178,6.940846
ILMN_1722622,0.954431,4.428023e-28,6.209121e-24,8.108898,7.154468
ILMN_2336781,1.768847,2.794766e-25,2.939186e-21,10.845100,9.076252
ILMN_2334296,1.166558,5.969333e-25,5.022239e-21,8.573292,7.406734
ILMN_2406501,1.271828,4.760478e-23,3.337650e-19,8.854271,7.582443
ILMN_1731224,0.496965,1.002048e-22,5.848794e-19,8.337547,7.840583
ILMN_1800091,1.311577,1.145891e-22,5.848794e-19,8.720411,7.408835
ILMN_1733270,0.472350,1.251317e-22,5.848794e-19,7.402697,6.930347


## 6. Probe → Gene Symbol + Aggregazione

In [24]:
def probes_to_genes(de_df, probe_gene_map):
    """Converte DE probe-level → gene-level (best probe per gene)."""
    if de_df.empty:
        return de_df
    df = de_df.copy()
    df["gene_symbol"] = df.index.astype(str).map(probe_gene_map)
    n0 = len(df)
    df = df.dropna(subset=["gene_symbol"])
    df = df[~df["gene_symbol"].isin(["", "nan", "---", "NA"])]
    df = df.sort_values("pvalue").groupby("gene_symbol").first()
    print(f"  Mapped: {len(df)}/{n0} probes → {len(df)} geni unici")
    return df


print("Mapping IRI_vs_Control:")
de_main_genes = probes_to_genes(de_main, probe_to_gene)

print("\nMapping Ischemia_vs_Control:")
de_support_genes = probes_to_genes(de_support, probe_to_gene)

Mapping IRI_vs_Control:
  Mapped: 28794/42067 probes → 28794 geni unici

Mapping Ischemia_vs_Control:
  Mapped: 28794/42067 probes → 28794 geni unici


#### ATTENZIONE! 
I seed genes definitivi vengono dalla meta-analisi Fisher in R, non dal singolo dataset. Comunque il calcolo successivo dei seed genes è utile come sanity check confrontando i risultati con quello dell'articolo originale.

In [25]:
# Seed genes dal confronto principale
seed_genes = de_main_genes[
    (de_main_genes["padj"] < FDR_THRESHOLD) &
    (de_main_genes["log2FoldChange"].abs() > FC_THRESHOLD)
].copy()

print(f"\nSeed genes (|log2FC|>{FC_THRESHOLD}, FDR<{FDR_THRESHOLD}):")
print(f"  Totale: {len(seed_genes)}")
print(f"  Up:     {(seed_genes['log2FoldChange'] > 0).sum()}")
print(f"  Down:   {(seed_genes['log2FoldChange'] < 0).sum()}")

# Annota: il gene è alterato anche nella sola ischemia?
support_sig = de_support_genes[
    (de_support_genes["padj"] < FDR_THRESHOLD) &
    (de_support_genes["log2FoldChange"].abs() > FC_THRESHOLD)
]
seed_genes["ischemia_also"] = seed_genes.index.isin(support_sig.index)

print(f"\n  Alterati anche in ischemia:      {seed_genes['ischemia_also'].sum()}")
print(f"  Specifici della riperfusione:     {(~seed_genes['ischemia_also']).sum()}")

# %%
# Top 30 seed genes
print("\nTop 30 seed genes:")
display(seed_genes.sort_values("pvalue").head(30)[
    ["log2FoldChange", "padj", "mean_case", "mean_ctrl", "ischemia_also"]])


Seed genes (|log2FC|>0.379, FDR<0.05):
  Totale: 1269
  Up:     740
  Down:   529

  Alterati anche in ischemia:      651
  Specifici della riperfusione:     618

Top 30 seed genes:


,log2FoldChange,padj,mean_case,mean_ctrl,ischemia_also
gene_symbol,,,,,
LOC85389,1.638703,6.599092e-71,8.666714,7.028011,False
LOC85390,1.839263,7.650296e-65,8.819493,6.980230,False
CXCL2,1.917954,9.021215e-61,8.888767,6.970814,False
HSPA6,1.905736,1.207671e-54,8.886299,6.980563,False
DNAJA4,2.249539,1.867761e-54,10.106705,7.857167,False
KLF4,2.180290,1.569272e-51,9.526787,7.346497,False
RGS1,1.419350,5.198957e-51,8.452227,7.032877,False
DNAJB4,1.407566,6.514197e-47,9.049608,7.642042,False
SNORD3C,1.801156,2.253527e-45,8.905010,7.103855,False


## 7. Verifica geni chiave dall'articolo

In [26]:
key_genes = {
    # Ipossia
    "HIF1A": "hypoxia-inducible factor",
    "SOD2": "superoxide dismutase (anti-ossidante)",
    "HMOX1": "heme oxygenase (citoprotettivo)",
    "LDHA": "lactate dehydrogenase (glicolisi anaerobia)",
    "HSPA6": "HSP70 (heat shock protein)",
    "HSPA1A": "HSP70 (heat shock protein)",
    # Complemento/Coagulazione
    "C3": "complement C3",
    "CFB": "complement factor B",
    "C1QC": "complement C1q",
    "FGA": "fibrinogen alpha",
    "FGB": "fibrinogen beta",
    "FGG": "fibrinogen gamma",
    "SERPINE1": "PAI-1 (inibitore fibrinolisi)",
    # Infiammazione
    "IL6": "interleuchina 6",
    "TNF": "tumor necrosis factor",
    "NFKBIA": "NF-κB inhibitor alpha",
    "ICAM1": "intercellular adhesion molecule",
    "VCAM1": "vascular cell adhesion molecule",
    "S100A8": "calprotectin (DAMP)",
    "S100A9": "calprotectin (DAMP)",
    "TNFAIP3": "A20 (anti-infiammatorio)",
    # NOD-like / riperfusione
    "NLRP3": "inflammasome",
    "CASP1": "caspasi 1",
}

print(f"{'Gene':<12} {'log2FC':>8} {'padj':>12} {'In seed':>8}  Ruolo")
print("-" * 72)
for gene, desc in key_genes.items():
    if gene in de_main_genes.index:
        row = de_main_genes.loc[gene]
        in_seed = "✓" if gene in seed_genes.index else ""
        print(f"{gene:<12} {row['log2FoldChange']:>8.3f} {row['padj']:>12.2e} {in_seed:>8}  {desc}")
    else:
        print(f"{gene:<12} {'N/A':>8} {'N/A':>12} {'':>8}  {desc}")


Gene           log2FC         padj  In seed  Ruolo
------------------------------------------------------------------------
HIF1A           0.672     2.80e-09        ✓  hypoxia-inducible factor
SOD2            1.557     2.96e-29        ✓  superoxide dismutase (anti-ossidante)
HMOX1           1.353     1.29e-15        ✓  heme oxygenase (citoprotettivo)
LDHA            0.311     2.02e-15           lactate dehydrogenase (glicolisi anaerobia)
HSPA6           1.906     1.21e-54        ✓  HSP70 (heat shock protein)
HSPA1A          3.297     2.83e-41        ✓  HSP70 (heat shock protein)
C3              0.430     5.10e-13        ✓  complement C3
CFB             1.181     6.97e-22        ✓  complement factor B
C1QC            0.485     2.66e-10        ✓  complement C1q
FGA             1.799     5.70e-13        ✓  fibrinogen alpha
FGB             1.575     1.09e-06        ✓  fibrinogen beta
FGG             1.296     5.16e-12        ✓  fibrinogen gamma
SERPINE1        0.777     1.60e-08        ✓ 

In [27]:
de_main_genes.head(30)

,log2FoldChange,pvalue,t_statistic,mean_case,mean_ctrl,padj,comparison,dataset
gene_symbol,,,,,,,,
7A5,0.002923,9.008212e-01,0.125265,6.926516,6.923593,9.620825e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A1BG,0.026651,2.018917e-01,1.289221,6.965492,6.938842,4.608985e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A1CF,-0.032082,5.534422e-02,-1.954353,6.682866,6.714948,1.960230e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A26C3,0.022419,3.350071e-01,0.972072,6.952956,6.930537,6.159376e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A2BP1,-0.021313,2.232280e-01,-1.232141,6.816348,6.837660,4.892984e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A2LD1,-1.984850,1.767644e-12,-9.795339,7.955390,9.940240,1.325481e-10,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A2M,0.227249,3.012995e-02,2.219886,10.454262,10.227013,1.249862e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A3GALT2,-0.016073,4.312850e-01,-0.793271,6.829128,6.845201,7.010381e-01,IRI_vs_Control (T3_deceased vs T1_living),GSE43974
A4GALT,0.426935,9.996628e-10,7.140480,7.903951,7.477016,4.585912e-08,IRI_vs_Control (T3_deceased vs T1_living),GSE43974


## 8. Salvataggio

In [28]:
de_main_genes.to_csv(OUTPUT_DIR / "de_GSE43974_IRI_vs_Control.csv")
de_support_genes.to_csv(OUTPUT_DIR / "de_GSE43974_Ischemia_vs_Control.csv")
# I seed genes vengono dalla metanalisi
# seed_genes.to_csv(OUTPUT_DIR / "seed_genes_GSE43974.csv")

# Ranking (log2FC dal confronto principale)
ranking = de_main_genes["log2FoldChange"].dropna()
ranking = ranking[~ranking.index.duplicated(keep='first')]
ranking.to_csv(OUTPUT_DIR / "ranking_GSE43974.csv")

print(f"File salvati in {OUTPUT_DIR}/:")
print(f"  de_GSE43974_IRI_vs_Control.csv      ({len(de_main_genes)} geni)")
print(f"  de_GSE43974_Ischemia_vs_Control.csv  ({len(de_support_genes)} geni)")
# print(f"  seed_genes_GSE43974.csv              ({len(seed_genes)} geni)")
print(f"  ranking_GSE43974.csv                 ({len(ranking)} geni)")

File salvati in /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/output/fase1/:
  de_GSE43974_IRI_vs_Control.csv      (28794 geni)
  de_GSE43974_Ischemia_vs_Control.csv  (28794 geni)
  ranking_GSE43974.csv                 (28794 geni)


In [40]:
expr_for_r = expr[samples_for_r].copy()
expr_for_r.index = expr_for_r.index.map(probe_to_gene)

In [37]:
expr_for_r

,GSM1075191,GSM1075192,GSM1075201,GSM1075205,GSM1075214,GSM1075243,GSM1075268,GSM1075275,GSM1075298,GSM1075328,...,GSM1075658,GSM1075662,GSM1075669,GSM1075685,GSM1075686,GSM1075696,GSM1075700,GSM1075703,GSM1075717,GSM1075718
probe_id,,,,,,,,,,,,,,,,,,,,,
ILMN_1762337,6.932362,6.902982,6.931155,7.154700,6.809661,6.814623,6.852063,6.723861,6.957980,6.954656,...,6.838179,6.836489,6.936640,6.903443,6.863729,6.905122,6.948552,6.935545,6.997342,7.034329
ILMN_2055271,6.979315,6.917292,6.975867,6.931813,6.888559,7.056527,7.069438,6.948922,6.836466,6.838367,...,6.974004,7.113715,6.994221,6.999003,6.999085,6.994581,7.016569,7.137802,6.877419,6.858292
ILMN_1736007,6.743820,6.780763,6.868474,6.798980,6.618603,6.648552,6.758506,6.704110,6.643250,6.884251,...,6.510666,6.821850,6.761302,6.749508,6.578666,6.727906,6.707429,6.896323,6.650544,6.612297
ILMN_2383229,8.019295,8.715523,8.411815,6.830013,8.041595,8.945570,8.796230,8.559418,8.605826,8.306156,...,7.604827,7.806280,7.681139,8.639685,8.586723,8.818847,9.175351,7.868802,6.874840,7.807298
ILMN_1806310,8.280550,9.229780,9.181406,7.105974,8.595576,9.284668,9.140277,9.001259,9.151254,8.775131,...,8.058199,8.082119,8.101101,9.155241,9.122090,9.227612,9.672248,8.426546,7.035090,8.053186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ILMN_2371169,7.847351,7.562991,7.402872,8.149173,7.611281,7.397328,7.413759,7.397139,7.376047,7.589471,...,7.646851,7.903918,7.725233,7.280383,7.284512,7.479959,7.633251,8.187563,7.670982,7.849700
ILMN_1701875,8.304882,7.792846,7.900455,9.031474,7.936845,7.890096,8.005403,7.763186,8.111888,7.754053,...,8.264186,8.581234,8.290947,7.695031,7.722322,7.867170,8.118186,8.049109,8.184224,8.847795
ILMN_1786396,7.619728,7.752222,7.715680,7.801440,7.627168,7.926990,7.646280,7.490799,7.515164,7.682998,...,7.534111,7.534782,7.458412,7.575955,7.596940,7.836053,7.549148,7.586741,7.711906,7.745705


In [41]:
expr_for_r

,GSM1075191,GSM1075192,GSM1075201,GSM1075205,GSM1075214,GSM1075243,GSM1075268,GSM1075275,GSM1075298,GSM1075328,...,GSM1075658,GSM1075662,GSM1075669,GSM1075685,GSM1075686,GSM1075696,GSM1075700,GSM1075703,GSM1075717,GSM1075718
probe_id,,,,,,,,,,,,,,,,,,,,,
7A5,6.932362,6.902982,6.931155,7.154700,6.809661,6.814623,6.852063,6.723861,6.957980,6.954656,...,6.838179,6.836489,6.936640,6.903443,6.863729,6.905122,6.948552,6.935545,6.997342,7.034329
A1BG,6.979315,6.917292,6.975867,6.931813,6.888559,7.056527,7.069438,6.948922,6.836466,6.838367,...,6.974004,7.113715,6.994221,6.999003,6.999085,6.994581,7.016569,7.137802,6.877419,6.858292
A1BG,6.743820,6.780763,6.868474,6.798980,6.618603,6.648552,6.758506,6.704110,6.643250,6.884251,...,6.510666,6.821850,6.761302,6.749508,6.578666,6.727906,6.707429,6.896323,6.650544,6.612297
A1CF,8.019295,8.715523,8.411815,6.830013,8.041595,8.945570,8.796230,8.559418,8.605826,8.306156,...,7.604827,7.806280,7.681139,8.639685,8.586723,8.818847,9.175351,7.868802,6.874840,7.807298
A1CF,8.280550,9.229780,9.181406,7.105974,8.595576,9.284668,9.140277,9.001259,9.151254,8.775131,...,8.058199,8.082119,8.101101,9.155241,9.122090,9.227612,9.672248,8.426546,7.035090,8.053186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYX,7.847351,7.562991,7.402872,8.149173,7.611281,7.397328,7.413759,7.397139,7.376047,7.589471,...,7.646851,7.903918,7.725233,7.280383,7.284512,7.479959,7.633251,8.187563,7.670982,7.849700
ZYX,8.304882,7.792846,7.900455,9.031474,7.936845,7.890096,8.005403,7.763186,8.111888,7.754053,...,8.264186,8.581234,8.290947,7.695031,7.722322,7.867170,8.118186,8.049109,8.184224,8.847795
ZZEF1,7.619728,7.752222,7.715680,7.801440,7.627168,7.926990,7.646280,7.490799,7.515164,7.682998,...,7.534111,7.534782,7.458412,7.575955,7.596940,7.836053,7.549148,7.586741,7.711906,7.745705


In [43]:
sample_labels

,sample,condition
0,GSM1075191,Control
1,GSM1075192,Control
2,GSM1075201,Control
3,GSM1075205,Control
4,GSM1075214,Control
...,...,...
201,GSM1075696,IRI
202,GSM1075700,IRI
203,GSM1075703,IRI
204,GSM1075717,IRI


In [45]:
gene_to_best_probe

{'LOC85389': 'ILMN_1754332',
 'LOC85390': 'ILMN_1689294',
 'CXCL2': 'ILMN_1682636',
 'HSPA6': 'ILMN_1658383',
 'DNAJA4': 'ILMN_1776998',
 'KLF4': 'ILMN_1779857',
 'RGS1': 'ILMN_1656011',
 'DNAJB4': 'ILMN_2222317',
 'SNORD3C': 'ILMN_3241034',
 'SNORD3A': 'ILMN_3239574',
 'RNU1A3': 'ILMN_3245678',
 'HSPA1A': 'ILMN_1789074',
 'TNFAIP3': 'ILMN_1702691',
 'HSPH1': 'ILMN_1712888',
 'DNAJB1': 'ILMN_1775304',
 'RNU1-5': 'ILMN_3236653',
 'RNU1F1': 'ILMN_3240220',
 'ZFAND2A': 'ILMN_1694671',
 'SERPINH1': 'ILMN_1751028',
 'RNU1G2': 'ILMN_3244646',
 'RNU1-3': 'ILMN_3246273',
 'GLA': 'ILMN_1766637',
 'RRAD': 'ILMN_1683179',
 'SNORD3D': 'ILMN_3242315',
 'CXCL1': 'ILMN_1787897',
 'RGS16': 'ILMN_1808226',
 'DNAJA1': 'ILMN_1672496',
 'LRG1': 'ILMN_1805228',
 'HSPA7': 'ILMN_3235964',
 'C10orf10': 'ILMN_1767556',
 'IL8': 'ILMN_2184373',
 'SOCS3': 'ILMN_1781001',
 'HMGCS1': 'ILMN_1797728',
 'HSPCAL3': 'ILMN_1700810',
 'REG1A': 'ILMN_1802441',
 'ZNF165': 'ILMN_1806502',
 'ATF3': 'ILMN_1791346',
 'HIST2H2AA

In [46]:
probes_to_keep

['ILMN_1754332',
 'ILMN_1689294',
 'ILMN_1682636',
 'ILMN_1658383',
 'ILMN_1776998',
 'ILMN_1779857',
 'ILMN_1656011',
 'ILMN_2222317',
 'ILMN_3241034',
 'ILMN_3239574',
 'ILMN_3245678',
 'ILMN_1789074',
 'ILMN_1702691',
 'ILMN_1712888',
 'ILMN_1775304',
 'ILMN_3236653',
 'ILMN_3240220',
 'ILMN_1694671',
 'ILMN_1751028',
 'ILMN_3244646',
 'ILMN_3246273',
 'ILMN_1766637',
 'ILMN_1683179',
 'ILMN_3242315',
 'ILMN_1787897',
 'ILMN_1808226',
 'ILMN_1672496',
 'ILMN_1805228',
 'ILMN_3235964',
 'ILMN_1767556',
 'ILMN_2184373',
 'ILMN_1781001',
 'ILMN_1797728',
 'ILMN_1700810',
 'ILMN_1802441',
 'ILMN_1806502',
 'ILMN_1791346',
 'ILMN_2144426',
 'ILMN_1788874',
 'ILMN_3177333',
 'ILMN_2406501',
 'ILMN_1728677',
 'ILMN_1700727',
 'ILMN_1756417',
 'ILMN_1676984',
 'ILMN_2188333',
 'ILMN_1715508',
 'ILMN_2214678',
 'ILMN_1674236',
 'ILMN_3242900',
 'ILMN_1717313',
 'ILMN_1683129',
 'ILMN_1798256',
 'ILMN_1732538',
 'ILMN_1716195',
 'ILMN_2072296',
 'ILMN_1752273',
 'ILMN_2334296',
 'ILMN_1768973

In [44]:
# =============================================================
# Export matrice espressione per R/MUUMI
# =============================================================
# Campioni IRI + Control (quelli usati nel confronto principale)
samples_for_r = [s for s in living_t1 + deceased_t3 if s in expr.columns]

# Converti a gene-level usando la STESSA probe selezionata nella DE
# (best probe per p-value, coerente con probes_to_genes)
best_probes_map = de_main.copy()
best_probes_map["gene_symbol"] = best_probes_map.index.map(probe_to_gene)
best_probes_map = best_probes_map.dropna(subset=["gene_symbol"])
best_probes_map = best_probes_map[~best_probes_map["gene_symbol"].isin(["", "nan", "---", "NA"])]
best_probes_map = best_probes_map.sort_values("pvalue").drop_duplicates(subset="gene_symbol", keep="first")
gene_to_best_probe = dict(zip(best_probes_map["gene_symbol"], best_probes_map.index))

# Seleziona solo le best probes dalla matrice di espressione
probes_to_keep = list(gene_to_best_probe.values())
expr_for_r = expr.loc[expr.index.isin(probes_to_keep), samples_for_r].copy()
expr_for_r.index = expr_for_r.index.map({v: k for k, v in gene_to_best_probe.items()})

# Batch labels
batch_labels = pd.DataFrame({
    "sample": samples_for_r,
    "batch": "GSE43974"
})

# Sample labels
sample_labels = pd.DataFrame({
    "sample": samples_for_r,
    "condition": ["Control" if s in living_t1 else "IRI" for s in samples_for_r]
})

# Salva
expr_for_r.to_csv(OUTPUT_DIR / "expr_GSE43974.csv")
batch_labels.to_csv(OUTPUT_DIR / "batch_labels_GSE43974.csv", index=False)
sample_labels.to_csv(OUTPUT_DIR / "sample_labels_GSE43974.csv", index=False)

print(f"Export per R/MUUMI:")
print(f"  expr_GSE43974.csv           ({expr_for_r.shape[0]} geni × {expr_for_r.shape[1]} campioni)")
print(f"  batch_labels_GSE43974.csv   ({len(batch_labels)} campioni)")
print(f"  sample_labels_GSE43974.csv  ({len(sample_labels)} campioni: "
      f"{(sample_labels['condition']=='Control').sum()} Control, "
      f"{(sample_labels['condition']=='IRI').sum()} IRI)")

Export per R/MUUMI:
  expr_GSE43974.csv           (28794 geni × 206 campioni)
  batch_labels_GSE43974.csv   (206 campioni)
  sample_labels_GSE43974.csv  (206 campioni: 37 Control, 169 IRI)


### Un chiarimento importante:
Si salvano i file per-dataset (expr_GSE43974.csv, non expr_microarray.csv), perché la matrice unificata expr_microarray.csv verrà assemblata solo quando si saranno completati tutti i dataset microarray Cat. A. A quel punto si concatenano i singoli expr_GSEXXXXX.csv e si uniscono i rispettivi batch/sample labels.

In [30]:
print("\n" + "=" * 60)
print("RIEPILOGO — GSE43974")
print("=" * 60)
print(f"Campioni totali GEO:  554")
print(f"Rimossi (rene destro): 31")
print(f"Campioni analizzati:  {len(meta)}")
print(f"  Control (Living_T1):    {len(living_t1)}")
print(f"  IRI (Deceased_T3):      {len(deceased_t3)}")
print(f"  Ischemia (Deceased_T1): {len(deceased_t1)}")
print(f"Probes dopo filtro:       {len(expr)}")
print(f"Geni mappati:             {len(de_main_genes)}")
print(f"Seed genes:               {len(seed_genes)} "
      f"({(seed_genes['log2FoldChange']>0).sum()}↑ "
      f"{(seed_genes['log2FoldChange']<0).sum()}↓)")
print(f"\nPronto per aggiungere prossimo dataset.")


RIEPILOGO — GSE43974
Campioni totali GEO:  554
Rimossi (rene destro): 31
Campioni analizzati:  523
  Control (Living_T1):    37
  IRI (Deceased_T3):      169
  Ischemia (Deceased_T1): 120
Probes dopo filtro:       42067
Geni mappati:             28794
Seed genes:               1269 (740↑ 529↓)

Pronto per aggiungere prossimo dataset.


## test

In [31]:
# ==============================================================================
# 9. MACHINE LEARNING: Predizione della DGF dal trascrittoma pre-riperfusione (T1)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

print("PREPARAZIONE DATI ML...")
# Selezioniamo solo i campioni di donatori deceduti (BD e DCD) al timepoint T1
# e per i quali abbiamo effettivamente il dato clinico DGF
mask_t1_deceased = (meta['timepoint'] == 1.0) & (meta['donor_type'].isin(['BD', 'DCD'])) & meta['dgf'].notna()
valid_samples = meta[mask_t1_deceased].index.intersection(expr.columns)

# X: feature matrix (Campioni sulle righe, Sonde sulle colonne)
X = expr[valid_samples].T

# y: target vector (0.0 = No DGF, 1.0 = DGF)
y = meta.loc[valid_samples, 'dgf'].astype(int)

print(f"Campioni T1 totali utilizzabili: {len(valid_samples)}")
print(f"Distribuzione classi DGF:\\n{y.value_counts().to_string()}\\n")

# --- Creazione Pipeline ---
# Step 1: SelectKBest (riduce le feature da 42k a 100 per limitare l'overfitting)
# Step 2: Classificatore (pesato per bilanciare la lieve asimmetria delle classi)

pipeline_rf = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif, k=100)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=0, class_weight='balanced'))
])

pipeline_lr = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif, k=100)),
    ('classifier', LogisticRegression(random_state=0, class_weight='balanced', max_iter=1000))
])

# --- Cross Validation (5-Fold Stratificata) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = ['accuracy', 'roc_auc', 'f1']

print("Esecuzione Cross-Validation (Random Forest)...")
cv_results_rf = cross_validate(pipeline_rf, X, y, cv=cv, scoring=scoring)

print("Esecuzione Cross-Validation (Logistic Regression)...")
cv_results_lr = cross_validate(pipeline_lr, X, y, cv=cv, scoring=scoring)

# Stampa dei risultati
print("\n=== RISULTATI CROSS-VALIDATION (5 Folds) ===")
print(f"Random Forest      -> ROC-AUC: {np.mean(cv_results_rf['test_roc_auc']):.3f} (±{np.std(cv_results_rf['test_roc_auc']):.3f}) | F1-Score: {np.mean(cv_results_rf['test_f1']):.3f}")
print(f"Logistic Regress.  -> ROC-AUC: {np.mean(cv_results_lr['test_roc_auc']):.3f} (±{np.std(cv_results_lr['test_roc_auc']):.3f}) | F1-Score: {np.mean(cv_results_lr['test_f1']):.3f}")

# --- Estrazione dei Biomarcatori Predittivi ---
# Ora addestriamo il Random Forest sull'INTERO dataset per capire quali 
# geni ha ritenuto più importanti per separare chi avrà DGF da chi non l'avrà
pipeline_rf.fit(X, y)

# Recuperiamo le feature selezionate e la loro importanza
selector = pipeline_rf.named_steps['feature_selection']
rf_model = pipeline_rf.named_steps['classifier']

selected_indices = selector.get_support(indices=True)
selected_probes = X.columns[selected_indices]
importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'probe_id': selected_probes,
    'importance': importances
})

# Mappiamo le sonde sui nomi dei geni usando il dizionario 'probe_to_gene' 
# che hai generato precedentemente nel notebook
if 'probe_to_gene' in locals() or 'probe_to_gene' in globals():
    feature_importance_df['gene_symbol'] = feature_importance_df['probe_id'].map(probe_to_gene)
else:
    feature_importance_df['gene_symbol'] = "Unknown"

feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

print("\n=== TOP 15 GENI PREDITTIVI PER DGF NEL TIMEPOINT T1 ===")
display(feature_importance_df.head(15))

PREPARAZIONE DATI ML...
Campioni T1 totali utilizzabili: 111
Distribuzione classi DGF:\ndgf
0    69
1    42\n
Esecuzione Cross-Validation (Random Forest)...
Esecuzione Cross-Validation (Logistic Regression)...

=== RISULTATI CROSS-VALIDATION (5 Folds) ===
Random Forest      -> ROC-AUC: 0.543 (±0.072) | F1-Score: 0.155
Logistic Regress.  -> ROC-AUC: 0.604 (±0.062) | F1-Score: 0.495

=== TOP 15 GENI PREDITTIVI PER DGF NEL TIMEPOINT T1 ===


,probe_id,importance,gene_symbol
1,ILMN_1685413,0.038521,ALG8
66,ILMN_2314244,0.032189,LRRC23
97,ILMN_2140999,0.023553,ZFYVE16
44,ILMN_1668554,0.020546,LOC642772
19,ILMN_1817180,0.020526,NaN
57,ILMN_1717423,0.019217,LOC652476
59,ILMN_1767664,0.018148,LOC653431
33,ILMN_3192645,0.017593,KRT18P42
56,ILMN_1689188,0.017554,LOC650874
88,ILMN_2352023,0.017475,RIPK5


In [32]:
# ==============================================================================
# 10. MACHINE LEARNING: Predizione della DGF dal trascrittoma post-riperfusione (T3)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

print("PREPARAZIONE DATI ML (TIMEPOINT T3)...")
# Selezioniamo solo i campioni di donatori deceduti (BD e DCD) al timepoint T3 (post-riperfusione)
# per i quali abbiamo il dato clinico DGF
mask_t3_deceased = (meta['timepoint'] == 3.0) & (meta['donor_type'].isin(['BD', 'DCD'])) & meta['dgf'].notna()
valid_samples_t3 = meta[mask_t3_deceased].index.intersection(expr.columns)

# X: feature matrix (Campioni sulle righe, Sonde sulle colonne)
X_t3 = expr[valid_samples_t3].T

# y: target vector (0.0 = No DGF, 1.0 = DGF)
y_t3 = meta.loc[valid_samples_t3, 'dgf'].astype(int)

print(f"Campioni T3 totali utilizzabili: {len(valid_samples_t3)}")
print(f"Distribuzione classi DGF nel T3:\n{y_t3.value_counts().to_string()}\n")

# --- Creazione Pipeline ---
# Step 1: SelectKBest per limitare l'overfitting selezionando le 100 sonde più associate alla DGF
# Step 2: Classificatore pesato per bilanciare le classi

pipeline_rf_t3 = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif, k=100)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

pipeline_lr_t3 = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif, k=100)),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])

# --- Cross Validation (5-Fold Stratificata) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'roc_auc', 'f1']

print("Esecuzione Cross-Validation T3 (Random Forest)...")
cv_results_rf_t3 = cross_validate(pipeline_rf_t3, X_t3, y_t3, cv=cv, scoring=scoring)

print("Esecuzione Cross-Validation T3 (Logistic Regression)...")
cv_results_lr_t3 = cross_validate(pipeline_lr_t3, X_t3, y_t3, cv=cv, scoring=scoring)

# Stampa dei risultati
print("\n=== RISULTATI CROSS-VALIDATION T3 (5 Folds) ===")
print(f"Random Forest      -> ROC-AUC: {np.mean(cv_results_rf_t3['test_roc_auc']):.3f} (±{np.std(cv_results_rf_t3['test_roc_auc']):.3f}) | F1-Score: {np.mean(cv_results_rf_t3['test_f1']):.3f}")
print(f"Logistic Regress.  -> ROC-AUC: {np.mean(cv_results_lr_t3['test_roc_auc']):.3f} (±{np.std(cv_results_lr_t3['test_roc_auc']):.3f}) | F1-Score: {np.mean(cv_results_lr_t3['test_f1']):.3f}")

# --- Estrazione dei Biomarcatori Predittivi T3 ---
# Addestriamo il Random Forest sull'INTERO dataset T3 per l'interpretazione biologica
pipeline_rf_t3.fit(X_t3, y_t3)

# Recuperiamo le feature e l'importanza
selector_t3 = pipeline_rf_t3.named_steps['feature_selection']
rf_model_t3 = pipeline_rf_t3.named_steps['classifier']

selected_indices_t3 = selector_t3.get_support(indices=True)
selected_probes_t3 = X_t3.columns[selected_indices_t3]
importances_t3 = rf_model_t3.feature_importances_

feature_importance_t3_df = pd.DataFrame({
    'probe_id': selected_probes_t3,
    'importance': importances_t3
})

# Mapping Probe -> Gene
if 'probe_to_gene' in locals() or 'probe_to_gene' in globals():
    feature_importance_t3_df['gene_symbol'] = feature_importance_t3_df['probe_id'].map(probe_to_gene)
else:
    feature_importance_t3_df['gene_symbol'] = "Unknown"

feature_importance_t3_df = feature_importance_t3_df.sort_values(by='importance', ascending=False)

print("\n=== TOP 15 GENI PREDITTIVI PER DGF NEL TIMEPOINT T3 ===")
display(feature_importance_t3_df.head(15))

PREPARAZIONE DATI ML (TIMEPOINT T3)...
Campioni T3 totali utilizzabili: 169
Distribuzione classi DGF nel T3:
dgf
1    89
0    80

Esecuzione Cross-Validation T3 (Random Forest)...
Esecuzione Cross-Validation T3 (Logistic Regression)...

=== RISULTATI CROSS-VALIDATION T3 (5 Folds) ===
Random Forest      -> ROC-AUC: 0.718 (±0.104) | F1-Score: 0.638
Logistic Regress.  -> ROC-AUC: 0.654 (±0.075) | F1-Score: 0.635

=== TOP 15 GENI PREDITTIVI PER DGF NEL TIMEPOINT T3 ===


,probe_id,importance,gene_symbol
46,ILMN_1713841,0.025073,LNX1
35,ILMN_1842797,0.023749,NaN
97,ILMN_1795228,0.022969,ZFAND5
62,ILMN_1704870,0.022082,PGLYRP1
78,ILMN_2406299,0.021824,SEMA3B
64,ILMN_1736670,0.021619,PPP1R3C
77,ILMN_1737709,0.019791,RPL10L
47,ILMN_3181328,0.019710,LOC100130179
94,ILMN_1677877,0.018806,UBE2L3
87,ILMN_2318643,0.018253,TGIF1
